# SoundStream Demo

In [ ]:
prompt = 'Hello world' # insert your promt here

### Cloning repo

In [ ]:
import os

if not os.path.exists('src'):
    !git clone https://github.com/arslan-yam/SoundStream.git
    %cd SoundStream

### Installing dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q torchcodec gdown
!pip install gdown
!pip install IPython

import sys
import subprocess

if sys.platform.startswith('linux'):
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)
    %pip install -q torchcodec
    
import gdown
import torch, torchaudio
from IPython.display import Audio, display
import urllib.request

### Downloading weights

In [ ]:
os.makedirs('checkpoints', exist_ok=True)
best_model_path = "saved/tts/model_best.pth"
file_id = "1XV94sTljhD_l76AYl57aT0UtVvwsF34A"
os.makedirs("saved/tts", exist_ok=True)
gdown.download(id=file_id, output=best_model_path, quiet=False)

### Generate speech

In [ ]:
from transformers import AutoTokenizer
from src.model.tts_model import TTSModel

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = TTSModel().to(device).eval()
ckpt = torch.load("saved/tts/model_best.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt["state_dict"], strict=False)

ids = tokenizer.encode(prompt, add_special_tokens=False)
text_ids = torch.tensor(ids, device=device).unsqueeze(0)
codes = model.generate(text_ids, max_tokens=500, temperature=0.8)
audio = model.decode_audio(codes).squeeze().cpu().clamp(-1, 1).numpy()
print(f"Generated {len(audio)/16000}s")
display(Audio(audio, rate=16000))